## ライブラリの読み込み

ライブラリを読み込みます．

In [2]:
import numpy as np

from genriesz import (
    grr_ate,
    SquaredGenerator,
    UKLGenerator,
    PolynomialBasis,
    TreatmentInteractionBasis,
    RBFRandomFourierBasis,
    KNNCatchmentBasis,
)

rng = np.random.default_rng(0)

## データの生成

複雑な回帰モデルに従うデータを生成します．  
$$ Y = f(D, X) + \epsilon $$
ここで，
- $D\in \{1, 0\}$は処置変数
- $X$は共変量
- $Y$はアウトカム
- $f$は未知の複雑な関数
- $\epsilon$は誤差項
です．平均処置効果は
$$ \theta = E[f(1, X) - f(0, X)] $$
として定義されます．

真の平均処置効果は$3$です．

In [5]:
# Data-generating process
## ATEの計算のために（分析では使わない）大量のデータを用意します
n_sim = 100000
## 実際に分析に使うデータを用意します．
n = 1000
d_z = 5

Z = rng.normal(size=(n, d_z))

# Treatment assignment: e(Z) = sigmoid(a'Z)
logits = 0.7 * Z[:, 0] - 0.3 * Z[:, 1]
e = 1.0 / (1.0 + np.exp(-logits))
D = rng.binomial(1, e, size=n).astype(float)

# Potential outcomes (constant effect for simplicity)
tau = 3.0
beta0 = np.random.uniform(0, 1, 4)
mu0 = beta0[0] * Z[:, 0] + beta0[1] * Z[:, 1] ** 2 + beta0[2] * Z[:, 0] * Z[:, 1] + beta0[3] * np.sin(Z[:, 2])

beta1 = np.random.uniform(0, 1, 2)
mu1 = beta1[0] * Z[:, 0] + beta1[1] * Z[:, 1] ** 2

mu0 = mu0 / np.mean(mu0)
mu1 = mu1 / np.mean(mu1) * (1 + tau)

Y0 = mu0 + rng.normal(scale=1.0, size=n)
Y1 = mu1 + rng.normal(scale=1.0, size=n)

Y = D * Y1 + (1.0 - D) * Y0

# Regressor matrix X = [D, Z...]
X = np.column_stack([D, Z])

Y = Y[:n]
X = X[:n]
print("X shape:", X.shape, "Y shape:", Y.shape)

X shape: (1000, 6) Y shape: (1000,)


## 回帰調整推定量・IPW推定量・AIPW推定量・TMLEによる推定（二次関数基底）

回帰関数と傾向スコアの推定に，二次の多項式基底（線形ではなく二次関数でフィットする）を用います．  
回帰調整推定量・IPW推定量・AIPW推定量・TMLEによる推定を実行し，それぞれで推定結果を返します．  

In [4]:
# Basis on Z, then interact with D (ATE-friendly)
psi = PolynomialBasis(degree=2, include_bias=True)
phi = TreatmentInteractionBasis(base_basis=psi)

# Generator: Squared loss (always safe / no domain constraints)
gen = SquaredGenerator(C=0.0).as_generator()

res_poly = grr_ate(
    X=X,
    Y=Y,
    basis=phi,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_poly.summary_text())


ATE estimates (n=1000)
alpha=0.05 | null=0.0
diagnostics: max_abs_smd_unweighted=0.7153645757945919, max_abs_smd_weighted=0.02558126061975452, ess_treated=402.6245935618626, ess_control=383.6878723951534

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 2.97912      0.167585         [ 2.65066,  3.30758]           0
RW                 3.01777      0.408751         [ 2.21663,  3.81891]    1.55e-13
ARW                2.97912      0.182363         [ 2.62169,  3.33654]           0
TMLE               2.97912      0.182363         [ 2.62169,  3.33654]           0


真の平均処置効果は$3$なので，どれもよく推定できています．

## 回帰調整推定量・IPW推定量・AIPW推定量・TMLEによる推定（再生核ヒルベルト空間）

回帰関数と傾向スコアの推定に，再生核ヒルベルト空間（動径ガウスカーネル）を用います．  
回帰調整推定量・IPW推定量・AIPW推定量・TMLEによる推定を実行し，それぞれで推定結果を返します．  

In [40]:
psi_rff = RBFRandomFourierBasis(
    n_features=500,
    sigma=1.0,
    standardize=True,
    random_state=0,
)
phi_rff = TreatmentInteractionBasis(base_basis=psi_rff)

res_rff = grr_ate(
    X=X,
    Y=Y,
    basis=phi_rff,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_rff.summary_text())


ATE estimates (n=1000)
alpha=0.05 | null=0.0
diagnostics: max_abs_smd_unweighted=0.6662380087434316, max_abs_smd_weighted=0.2443503805709245, ess_treated=445.4983650116201, ess_control=432.6136266240531

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                  4.4961      0.265099         [ 3.97651,  5.01568]           0
RW                 5.23514      0.804863         [ 3.65764,  6.81264]     7.8e-11
ARW                4.10464      0.568956         [ 2.98951,  5.21978]    5.42e-13
TMLE                4.1691      0.567248         [ 3.05732,  5.28089]    1.99e-13


真の平均処置効果は$3$なので，この場合はあまりうまく推定できていません．

## 回帰調整推定量・IPW推定量・AIPW推定量・TMLEによる推定（ニューラルネットワーク）

回帰関数と傾向スコアの推定に，ニューラルネットワークを用います．  
回帰調整推定量・IPW推定量・AIPW推定量・TMLEによる推定を実行し，それぞれで推定結果を返します．  

In [50]:
import torch
from genriesz.torch_basis import MLPEmbeddingNet, TorchEmbeddingBasis

torch.manual_seed(0)

net = MLPEmbeddingNet(input_dim=X.shape[1], hidden_dims=(64,), output_dim=32)
# (Optional) Train net here on a separate task.
# For a lightweight demo, we skip training and just use the random initialization.
nn_basis = TorchEmbeddingBasis(net, include_bias=True, device="cpu")

res_nn = grr_ate(
    X=X,
    Y=Y,
    basis=nn_basis,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_nn.summary_text())

ATE estimates (n=1000)
alpha=0.05 | null=0.0
diagnostics: max_abs_smd_unweighted=0.6662380087434316, max_abs_smd_weighted=0.11310789974578712, ess_treated=408.804419100671, ess_control=416.27665102482666

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 2.20889        0.1575          [ 1.9002,  2.51758]           0
RW                 2.25174      0.571085         [ 1.13243,  3.37104]    8.05e-05
ARW                2.36108      0.383275         [ 1.60987,  3.11229]    7.26e-10
TMLE               2.36555      0.382785          [ 1.6153,  3.11579]    6.42e-10


真の平均処置効果は$3$なので，この場合はまあまあうまく推定できています．